# Diffusion Kernel Assessment

The goal of this notebook is to walkthrough the "algo pipeline", raw data --> affinity --> diffusion kernel --> Laplacian --> spectral embedding --> ALC. We also look to visualise the how edges change through the diffusion process, as well as, the likelihood. Additionally, we use ARI as the measure of performance, perturb the data with noise to see impact, and perform an experimental sweep at a small scale. 

## Preamble 

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import scipy.linalg
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score

from clustering.spectral import dist_mat, knn_affinity
from clustering.alc import ALCAdapter
from data.generators import make_blobs_convex, make_two_moons, make_rings

import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

from scipy.linalg import expm

## Algorithmic Pipeline

### Raw Data Generation

In [ ]:
blobs, blobs_labels = make_blobs_convex(N=500, num_clusters=7, n_features=5000)
rings, rings_labels = make_rings(n_points=500, n_circles=3, noise_std=0.5)
moons, moons_labels = make_two_moons(n_samples=500, noise=0.25)

print(blobs.shape, rings.shape, moons.shape)

In [ ]:
plt.scatter(x=moons[:,0], y=moons[:,1], c=moons_labels)

### Affinity Matrix Construction 

In [ ]:
# ---------------- Distance matrix ----------------
def dist_mat(X):
    """Compute pairwise Euclidean distances (N x N)."""
    return cdist(X, X, "euclidean")

# ---------------- k-nn affinity matrix ----------------
def knn_affinity(dist_matrix, k, sigma):
    """
    Build a symmetric k-NN Gaussian affinity matrix from a precomputed
    pairwise distance matrix.

    Parameters
    ----------
    dist_matrix : (n, n) ndarray  — pairwise Euclidean distances, zero diagonal.
    k           : int             — number of nearest neighbours per point.
    sigma       : float           — Gaussian kernel bandwidth.

    Returns
    -------
    A : (n, n) ndarray  — symmetric, non-negative, zero diagonal.
    """
    n = dist_matrix.shape[0]

    # Fully connected fallback (backward-compatible)
    if k >= n - 1:
        A = np.exp(-(dist_matrix ** 2 / (2.0 * sigma ** 2)))
        np.fill_diagonal(A, 0.0)
        return A

    # Step 1 — k nearest neighbours per row (skip self at index 0)
    sorted_idx = np.argsort(dist_matrix, axis=1)
    nn_idx     = sorted_idx[:, 1:k + 1]          # (n, k)

    # Step 2 — directed adjacency mask
    nn_mask      = np.zeros((n, n), dtype=bool)
    rows         = np.repeat(np.arange(n), k)
    nn_mask[rows, nn_idx.ravel()] = True

    # Step 3 — OR-symmetrisation
    sym_mask = nn_mask | nn_mask.T

    # Step 4-5 — Gaussian weights applied to retained edges only
    W = np.exp(-(dist_matrix ** 2 / (2.0 * sigma ** 2)))
    A = W * sym_mask
    np.fill_diagonal(A, 0.0)
    return A


In [ ]:
# Finding Distance Matrices
blobs_dist_mat = dist_mat(blobs)
rings_dist_mat = dist_mat(rings)
moons_dist_mat = dist_mat(moons)

# Finding k-nn affinity matrices
blobs_affinity = knn_affinity(blobs_dist_mat, k=10, sigma=10)
rings_affinity = knn_affinity(rings_dist_mat, k=10, sigma=0.5)
moons_affinity = knn_affinity(moons_dist_mat, k=10, sigma=0.1)
rings_aff_full = knn_affinity(rings_dist_mat, k=1000, sigma=0.5)


### Diffusion Kernel 

In [ ]:
def diffusion_kernel(affinity, t):
    diff_K = expm(-t * affinity) # matrix exponential *not* elementwise operator
    return(diff_K)

In [ ]:
# Finding diffusion matrices
blobs_diff_1 = diffusion_kernel(blobs_affinity, t=3)
blobs_diff_2 = diffusion_kernel(blobs_affinity, t=100)
rings_diff_1 = diffusion_kernel(rings_affinity, t=3)
rings_diff_2 = diffusion_kernel(rings_affinity, t=100)
moons_diff_1 = diffusion_kernel(moons_affinity, t=2)
moons_diff_2 = diffusion_kernel(moons_affinity, t=100)

### Laplacian Construction

In [ ]:
def build_laplacian(A, laplacian_type):
    """
    Build a graph Laplacian from affinity matrix A.

    Parameters
    ----------
    A : (n, n) ndarray
        Symmetric non-negative affinity matrix.
    laplacian_type : {"unnorm", "sym", "rw"}
        Laplacian formulation to use.

    Returns
    -------
    L : (n, n) ndarray
    d : (n,) ndarray  — degree vector
    """
    n = A.shape[0]
    d = A.sum(axis=1)
    I = np.eye(n)

    if laplacian_type == "unnorm":
        D = np.diag(d)
        L = D - A

    elif laplacian_type == "sym":
        D_inv_sqrt = np.diag(1.0 / np.sqrt(d))
        L = I - D_inv_sqrt @ A @ D_inv_sqrt

    elif laplacian_type == "rw":
        D_inv = np.diag(1.0 / d)
        L = I - D_inv @ A

    else:
        raise ValueError(f"Unknown laplacian_type '{laplacian_type}'. Use 'unnorm', 'sym', or 'rw'.")

    return L, d

In [ ]:
# Finding the Laplacians

# BLOBS
blobs_lap_raw = build_laplacian(blobs_affinity, "unnorm")
blobs_lap_sym = build_laplacian(blobs_affinity, "sym")
blobs_lap_rw  = build_laplacian(blobs_affinity, "rw")

# RINGS
rings_lap_raw = build_laplacian(rings_affinity, "unnorm")
rings_lap_sym = build_laplacian(rings_affinity, "sym")
rings_lap_rw  = build_laplacian(rings_affinity, "rw")

# MOONS
moons_lap_raw = build_laplacian(moons_affinity, "unnorm")
moons_lap_sym = build_laplacian(moons_affinity, "sym")
moons_lap_rw  = build_laplacian(moons_affinity, "rw")


### Spectral Embedding


In [ ]:
# Finding eigenvectors of matrix

# Symmetric Matrices - Blobs
blobs_raw_eig_val, blobs_raw_eig_vec = scipy.linalg.eigh(blobs_lap_raw[0])
blobs_sym_eig_val, blobs_sym_eig_vec = scipy.linalg.eigh(blobs_lap_sym[0])

# General Matrix - Blobs
blobs_rw_eig_val, blobs_rw_eig_vec = scipy.linalg.eig(blobs_lap_rw[0])

# Symmetric Matrices - Rings
rings_raw_eig_val, rings_raw_eig_vec = scipy.linalg.eigh(rings_lap_raw[0])
rings_sym_eig_val, rings_sym_eig_vec = scipy.linalg.eigh(rings_lap_sym[0])

# General Matrix - Rings
rings_rw_eig_val, rings_rw_eig_vec = scipy.linalg.eig(rings_lap_rw[0])

# Symmetric Matrices - Moons
moons_raw_eig_val, moons_raw_eig_vec = scipy.linalg.eigh(moons_lap_raw[0])
moons_sym_eig_val, moons_sym_eig_vec = scipy.linalg.eigh(moons_lap_sym[0])

# General Matrix - Moons
moons_rw_eig_val, moons_rw_eig_vec = scipy.linalg.eig(moons_lap_rw[0])


In [ ]:
np.sort(moons_rw_eig_val.real.round(5))

In [ ]:
test


In [ ]:
alc = ALCAdapter()

In [ ]:
alc.fit_predict(rings_sym_eig_vec[:,:2],cn=None)

### Agglomerative Likelihood Clustering Execution


In [ ]:
import numpy as np
from numba import vectorize

@vectorize('float64(float64, float64)', nopython=True)
def lc(corr, size):
    if size <= 1.0:
        return 0.0
    if corr <= size:
        return 0.0
    if size**2 - corr <= 0.0:
        corr -= 1e-3
    return 0.5*(np.log(size/corr) + (size - 1.0) * np.log((size**2 - size) / (size**2 - corr)))

@vectorize('float64(float64, float64, float64, float64)', nopython=True)
def merge(corr_com, corr_neighbors, cross_neighbors, size):
    corr = corr_com + corr_neighbors + 2.0 * cross_neighbors
    if corr <= 0:
        return 0.0
    if size <= 1.0:
        return 0.0
    if corr <= size:
        return 0.0
    if size**2 - corr <= 0.0:
        corr -= 1e-3
    return 0.5*(np.log(size/corr) + (size - 1.0) * np.log((size**2 - size) / (size**2 - corr)))

def alc_dense_v2(G):
    N = len(G)
    communities = np.arange(N)
    tracker = np.arange(N)
    ns = np.ones(N)
    all_seen = False
    while not all_seen:
        all_seen = True
        while len(tracker) > 1:
            community = np.random.choice(tracker)
            community_neighbors = np.nonzero(G[community])[0]
            community_neighbors = community_neighbors[community_neighbors != community]
            A = lc(G[community,community], ns[community])
            B = lc(G[community_neighbors,community_neighbors], ns[community_neighbors])
            C = merge(G[community, community],
                      G[community_neighbors, community_neighbors],
                      G[community, community_neighbors],
                      ns[community] + ns[community_neighbors])
            scores = C - ( A + B )
            if np.all( scores <= 0 ):
                tracker = np.setdiff1d(tracker,community)
                continue
            pick = community_neighbors[np.argmax(scores)]
            pick_neighbors = np.nonzero(G[pick])[0]
            pick_neighbors = pick_neighbors[pick_neighbors != community]
            pick_neighbors = pick_neighbors[pick_neighbors != pick]
            G[community, pick_neighbors] += G[pick, pick_neighbors]
            G[pick_neighbors, community] += G[pick_neighbors, pick]
            G[community, community] = G[community,community] + G[pick, pick] + 2*G[pick, community]
            G[pick] = 0
            G[:,pick] = 0
            communities[communities == pick] = community
            ns[community] += ns[pick]
            ns[pick] = 0
            all_seen = False
            tracker = np.unique(communities)
            break
        if all_seen:
            return communities

In [ ]:
rings_sym_eig_val[:3]

In [ ]:
test = rings_sym_eig_vec[:,:3] @ np.diag(rings_sym_eig_val[:3]) @ np.transpose(rings_sym_eig_vec[:,:3])

In [ ]:
alc_dense_v2(test)

In [ ]:
alc = ALCAdapter()

# --- Blobs (7 clusters — take first 7 eigenvectors) ---
blobs_labels_raw = alc.fit_predict(blobs_raw_eig_vec[:, 1:7])
blobs_labels_sym = alc.fit_predict(blobs_sym_eig_vec[:, 1:7])
blobs_labels_rw  = alc.fit_predict(blobs_rw_eig_vec[:, 1:7].real)

print("Blobs")
print(f"  unnorm ARI: {adjusted_rand_score(blobs_labels, blobs_labels_raw):.4f}")
print(f"  sym    ARI: {adjusted_rand_score(blobs_labels, blobs_labels_sym):.4f}")
print(f"  rw     ARI: {adjusted_rand_score(blobs_labels, blobs_labels_rw):.4f}")

# --- Rings (3 clusters — take first 3 eigenvectors) ---
rings_labels_raw = alc.fit_predict(rings_raw_eig_vec[:, 1:3])
rings_labels_sym = alc.fit_predict(rings_sym_eig_vec[:, 1:3])
rings_labels_rw  = alc.fit_predict(rings_rw_eig_vec[:, 1:3].real)

print("\nRings")
print(f"  unnorm ARI: {adjusted_rand_score(rings_labels, rings_labels_raw):.4f}")
print(f"  sym    ARI: {adjusted_rand_score(rings_labels, rings_labels_sym):.4f}")
print(f"  rw     ARI: {adjusted_rand_score(rings_labels, rings_labels_rw):.4f}")

# --- Moons (2 clusters — take first 2 eigenvectors) ---
moons_labels_raw = alc.fit_predict(moons_raw_eig_vec[:, 1:2])
moons_labels_sym = alc.fit_predict(moons_sym_eig_vec[:, 1:2])
moons_labels_rw  = alc.fit_predict(moons_rw_eig_vec[:, 1:2].real)

print("\nMoons")
print(f"  unnorm ARI: {adjusted_rand_score(moons_labels, moons_labels_raw):.4f}")
print(f"  sym    ARI: {adjusted_rand_score(moons_labels, moons_labels_sym):.4f}")
print(f"  rw     ARI: {adjusted_rand_score(moons_labels, moons_labels_rw):.4f}")

## Performance Evaluation

## Visualations

In [ ]:
"""
Visualisation of diffusion kernel effect on graph structure.
Drop this into your notebook after computing A, K_t, and ground truth labels.

Requirements:
  - A: (n, n) affinity matrix from knn_affinity
  - K_t_moderate: (n, n) diffusion kernel at moderate t
  - K_t_large: (n, n) diffusion kernel at large t
  - y: (n,) ground truth labels
  - X: (n, 2) data matrix (for identifying boundary points)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


# ============================================================
# VISUALISATION 2: Sorted affinity matrix heatmaps
# ============================================================

def plot_affinity_heatmaps(A, K_t_mod, K_t_large, y, t_mod, t_large,
                           dataset_name="Dataset"):
    """
    Display three heatmaps side by side: the original affinity A,
    and the diffusion kernel K_t at two values of t.

    All matrices are sorted by ground truth label so that block-
    diagonal cluster structure is visible.

    Parameters
    ----------
    A          : (n, n) original affinity matrix.
    K_t_mod    : (n, n) diffusion kernel at moderate t.
    K_t_large  : (n, n) diffusion kernel at large t.
    y          : (n,) ground truth labels (integers).
    t_mod      : float, the moderate t value (for title).
    t_large    : float, the large t value (for title).
    dataset_name : string, for the figure title.
    """

    # ----------------------------------------------------------
    # Step 1: Build the sort order.
    #
    # np.argsort(y) returns indices that would sort y in ascending
    # order. All cluster-0 points come first, then cluster-1, etc.
    # We use this same permutation to reorder rows AND columns
    # of every matrix, so the heatmaps are visually comparable.
    # ----------------------------------------------------------
    sort_idx = np.argsort(y)
    y_sorted = y[sort_idx]

    # Apply the same permutation to rows and columns.
    # For a matrix M, the sorted version is M[sort_idx][:, sort_idx],
    # which reorders both axes by ground truth label.
    A_sorted         = A[sort_idx][:, sort_idx]
    K_mod_sorted     = K_t_mod[sort_idx][:, sort_idx]
    K_large_sorted   = K_t_large[sort_idx][:, sort_idx]

    # ----------------------------------------------------------
    # Step 2: Find cluster boundaries for annotation.
    #
    # After sorting, all points with label 0 are first, then label 1,
    # etc. We find the index where each label transitions to the next.
    # These boundaries are drawn as lines on the heatmap so you can
    # see where one cluster ends and the next begins.
    # ----------------------------------------------------------
    unique_labels = np.unique(y_sorted)
    boundaries = []
    for lbl in unique_labels[:-1]:  # no boundary after the last cluster
        # Find the last index with this label, boundary is one past it
        last_idx = np.where(y_sorted == lbl)[0][-1]
        boundaries.append(last_idx + 0.5)  # +0.5 centres the line between pixels

    # ----------------------------------------------------------
    # Step 3: Plot the three heatmaps.
    #
    # Each panel uses its own colour scale (vmin=0, vmax=panel max)
    # so that the internal contrast is visible regardless of the
    # absolute magnitude of the entries. The diffusion kernel can
    # have very different magnitudes to A.
    # ----------------------------------------------------------
    matrices = [A_sorted, K_mod_sorted, K_large_sorted]
    titles   = [
        "Original affinity A",
        f"Diffusion kernel K_t (t={t_mod})",
        f"Diffusion kernel K_t (t={t_large})",
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"{dataset_name} — Sorted affinity heatmaps", fontsize=12)

    for ax, mat, title in zip(axes, matrices, titles):
        # imshow displays the matrix as an image.
        # origin="upper" means row 0 is at the top (conventional matrix layout).
        # cmap="viridis" goes from dark (low) to bright (high).
        im = ax.imshow(mat, cmap="viridis", aspect="auto", origin="upper",
                       vmin=0, vmax=np.percentile(mat, 99))
        # The vmax is set to the 99th percentile rather than the absolute
        # max, because a few very large diagonal or near-diagonal entries
        # can wash out the rest of the colour scale.

        # Draw cluster boundary lines
        for b in boundaries:
            ax.axhline(b, color="white", linewidth=0.5, linestyle="--")
            ax.axvline(b, color="white", linewidth=0.5, linestyle="--")

        ax.set_title(title, fontsize=9)
        ax.set_xlabel("Point index (sorted by label)")
        ax.set_ylabel("Point index (sorted by label)")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


# ============================================================
# VISUALISATION 3: Row profiles for a boundary point
# ============================================================

def find_boundary_point(X, y, target_label=0):
    """
    Find the point in cluster target_label that is closest to any
    point in a different cluster. This is the "hardest" point —
    the one most likely to be confused by cross-cluster edges.

    Parameters
    ----------
    X            : (n, 2) data matrix.
    y            : (n,) ground truth labels.
    target_label : int, which cluster to pick the boundary point from.

    Returns
    -------
    boundary_idx : int, index of the boundary point in the original
                   (unsorted) data.
    """
    from scipy.spatial.distance import cdist

    # Points in the target cluster
    mask_target = (y == target_label)
    # Points in all other clusters
    mask_other  = ~mask_target

    X_target = X[mask_target]
    X_other  = X[mask_other]

    # Compute distances from each target point to every other-cluster point
    dists = cdist(X_target, X_other, "euclidean")

    # For each target point, find its minimum distance to any other-cluster point
    min_dists = dists.min(axis=1)

    # The boundary point is the one with the SMALLEST minimum distance
    # — it is the target-cluster point closest to the other cluster.
    boundary_in_target = np.argmin(min_dists)

    # Convert back to the original index
    target_indices = np.where(mask_target)[0]
    boundary_idx = target_indices[boundary_in_target]

    return boundary_idx


def plot_row_profiles(A, K_t_mod, K_t_large, y, boundary_idx,
                      t_mod, t_large, dataset_name="Dataset"):
    """
    For a single point (boundary_idx), plot its similarity to every
    other point under A, K_t at moderate t, and K_t at large t.

    Points on the x-axis are sorted by ground truth label, so you
    can see the within-cluster vs between-cluster similarity at a
    glance.

    Parameters
    ----------
    A             : (n, n) original affinity matrix.
    K_t_mod       : (n, n) diffusion kernel at moderate t.
    K_t_large     : (n, n) diffusion kernel at large t.
    y             : (n,) ground truth labels.
    boundary_idx  : int, the index of the point whose row we plot.
    t_mod         : float, moderate t value.
    t_large       : float, large t value.
    dataset_name  : string, for the figure title.
    """

    # ----------------------------------------------------------
    # Step 1: Sort by ground truth label (same ordering as Vis 2).
    # ----------------------------------------------------------
    sort_idx = np.argsort(y)
    y_sorted = y[sort_idx]

    # Extract the boundary point's row from each matrix,
    # then reorder by the sorted indices.
    row_A       = A[boundary_idx, sort_idx]
    row_K_mod   = K_t_mod[boundary_idx, sort_idx]
    row_K_large = K_t_large[boundary_idx, sort_idx]

    # ----------------------------------------------------------
    # Step 2: Find cluster boundaries and the point's own cluster.
    # ----------------------------------------------------------
    unique_labels = np.unique(y_sorted)
    boundaries = []
    for lbl in unique_labels[:-1]:
        last_idx = np.where(y_sorted == lbl)[0][-1]
        boundaries.append(last_idx + 0.5)

    point_label = y[boundary_idx]

    # ----------------------------------------------------------
    # Step 3: Find where the boundary point sits in the sorted order.
    #
    # This lets us mark it on the plot so you can see the point's
    # own position relative to its cluster.
    # ----------------------------------------------------------
    sorted_position = np.where(sort_idx == boundary_idx)[0][0]

    # ----------------------------------------------------------
    # Step 4: Plot three row profiles stacked vertically.
    #
    # Each panel shows the same point's row from a different matrix.
    # The x-axis is the sorted point index, the y-axis is the
    # affinity/kernel value. Vertical dashed lines mark cluster
    # boundaries. A red vertical line marks the point's own position.
    #
    # What to look for:
    #   - In the A panel: high values near the point's position
    #     (local neighbours), possibly some cross-boundary values.
    #   - In the K_t panels: the values should spread across the
    #     entire same-cluster block (diffusion has propagated along
    #     the manifold) and the cross-boundary values should shrink.
    # ----------------------------------------------------------
    rows   = [row_A, row_K_mod, row_K_large]
    titles = [
        f"Row of A (point {boundary_idx}, cluster {point_label})",
        f"Row of K_t, t={t_mod}",
        f"Row of K_t, t={t_large}",
    ]

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(
        f"{dataset_name} — Row profile for boundary point {boundary_idx} "
        f"(cluster {point_label})",
        fontsize=11
    )

    x_axis = np.arange(len(y))

    for ax, row, title in zip(axes, rows, titles):
        # Plot the row as a line. Using a thin line with slight
        # transparency so individual peaks are visible even when
        # there are many points.
        ax.plot(x_axis, row, linewidth=0.8, alpha=0.8, color="steelblue")

        # Fill under the curve lightly so the shape is visible
        ax.fill_between(x_axis, 0, row, alpha=0.15, color="steelblue")

        # Mark cluster boundaries
        for b in boundaries:
            ax.axvline(b, color="red", linewidth=0.8, linestyle="--",
                       alpha=0.6)

        # Mark the point's own position
        ax.axvline(sorted_position, color="black", linewidth=1.0,
                   linestyle=":", alpha=0.8)

        ax.set_ylabel("Affinity value")
        ax.set_title(title, fontsize=9)
        ax.set_ylim(bottom=0)

    axes[-1].set_xlabel("Point index (sorted by ground truth label)")

    # Add cluster label annotations at the bottom of the plot
    label_positions = []
    start = 0
    for lbl in unique_labels:
        count = np.sum(y_sorted == lbl)
        mid = start + count / 2
        label_positions.append((mid, lbl))
        start += count

    for mid, lbl in label_positions:
        axes[-1].text(mid, -0.05 * axes[-1].get_ylim()[1],
                      f"Cluster {lbl}", ha="center", fontsize=8,
                      color="red", alpha=0.7)

    plt.tight_layout()
    plt.show()


# ============================================================
# USAGE EXAMPLE
# ============================================================

"""
# --- Assuming you have already computed these: ---
# X, y          : data and ground truth labels
# A             : k-NN affinity matrix
# K_t_mod       : diffusion kernel at moderate t (e.g. t=1)
# K_t_large     : diffusion kernel at large t (e.g. t=5)

t_mod   = 1.0
t_large = 5.0

# --- Visualisation 2: Heatmaps ---
plot_affinity_heatmaps(A, K_t_mod, K_t_large, y,
                       t_mod, t_large, dataset_name="Two Moons")

# --- Visualisation 3: Row profiles ---
# Find the hardest point (closest to the other cluster)
boundary_idx = find_boundary_point(X, y, target_label=0)
print(f"Boundary point index: {boundary_idx}, label: {y[boundary_idx]}")

# Optional: verify visually that this point is near the boundary
fig, ax = plt.subplots(figsize=(6, 4))
for lbl in np.unique(y):
    mask = y == lbl
    ax.scatter(X[mask, 0], X[mask, 1], s=10, alpha=0.5, label=f"Cluster {lbl}")
ax.scatter(X[boundary_idx, 0], X[boundary_idx, 1],
           s=100, c="red", marker="x", zorder=5, label="Boundary point")
ax.legend(fontsize=8)
ax.set_title("Boundary point location")
plt.show()

# Plot the row profiles
plot_row_profiles(A, K_t_mod, K_t_large, y, boundary_idx,
                  t_mod, t_large, dataset_name="Two Moons")
"""

### --- Visualisation 2: Heatmaps ---

In [ ]:
plot_affinity_heatmaps(A=blobs_affinity, K_t_mod=blobs_diff_1, K_t_large=blobs_diff_2, y=blobs_labels,
                       t_mod="2", t_large="100", dataset_name="Blobs")

In [ ]:
plot_affinity_heatmaps(A=rings_affinity, K_t_mod=rings_diff_1, K_t_large=rings_diff_2, y=rings_labels,
                       t_mod="2", t_large="100", dataset_name="Rings")

In [ ]:
plot_affinity_heatmaps(A=moons_affinity, K_t_mod=moons_diff_1, K_t_large=moons_diff_2, y=moons_labels,
                       t_mod="2", t_large="100", dataset_name="Two Moons")

### --- Visualisation 3: Row profiles ---

In [ ]:
# --- Visualisation 3: Row profiles ---
# Find the hardest point (closest to the other cluster)
X=moons
y=moons_labels
boundary_idx_moons = find_boundary_point(X, y, target_label=0)
print(f"Boundary point index: {boundary_idx_moons}, label: {y[boundary_idx_moons]}")

# Optional: verify visually that this point is near the boundary
fig, ax = plt.subplots(figsize=(6, 4))
for lbl in np.unique(y):
    mask = y == lbl
    ax.scatter(X[mask, 0], X[mask, 1], s=10, alpha=0.5, label=f"Cluster {lbl}")
ax.scatter(X[boundary_idx_moons, 0], X[boundary_idx_moons, 1],
           s=100, c="red", marker="x", zorder=5, label="Boundary point")
ax.legend(fontsize=8)
ax.set_title("Boundary point location")
plt.show()

In [ ]:
# Plot the row profiles
plot_row_profiles(A=moons_affinity, K_t_mod=moons_diff_1, K_t_large=moons_diff_2, y=moons_labels, boundary_idx=boundary_idx_moons,
                  t_mod="2", t_large="100", dataset_name="Two Moons")

In [ ]:
def check_psd(M, name="Matrix"):
    """
    Check whether a matrix satisfies the conditions for
    positive semi-definiteness. Reports each condition
    separately so you can see exactly where a matrix fails.
    """
    n = M.shape[0]
    eigenvalues = np.linalg.eigvalsh(M)
    
    # Condition 1: Symmetry
    # A PSD matrix must be symmetric (M = M^T).
    # The quadratic form x^T M x is only guaranteed real-valued
    # when M is symmetric. If M is not symmetric, the eigenvalues
    # can be complex and PSD is not even defined.
    is_symmetric = np.allclose(M, M.T)
    
    # Condition 2: All eigenvalues non-negative
    # This is the core definition. M is PSD if and only if
    # x^T M x >= 0 for all x. For a symmetric matrix, this
    # holds if and only if all eigenvalues are >= 0.
    # Because: M = U diag(lambda) U^T, so
    # x^T M x = sum_j lambda_j (u_j^T x)^2.
    # Each (u_j^T x)^2 >= 0, so the sum is non-negative
    # if and only if every lambda_j >= 0.
    min_eigenvalue = eigenvalues.min()
    n_negative = (eigenvalues < -1e-10).sum()  # tolerance for numerics
    is_psd = min_eigenvalue >= -1e-10
    
    # Condition 3: Non-negative diagonal
    # A necessary (but not sufficient) condition. The diagonal
    # entry M_ii = e_i^T M e_i, where e_i is the i-th standard
    # basis vector. If M is PSD, then e_i^T M e_i >= 0, so
    # M_ii >= 0. If any diagonal entry is negative, the matrix
    # cannot be PSD. But non-negative diagonal does NOT guarantee
    # PSD — your affinity matrix demonstrates this.
    min_diagonal = np.diag(M).min()
    has_nonneg_diag = min_diagonal >= -1e-10

    print(f"=== PSD check for {name} (shape {M.shape}) ===")
    print(f"  Symmetric:             {is_symmetric}")
    print(f"  Min diagonal entry:    {min_diagonal:.6e}  (non-negative: {has_nonneg_diag})")
    print(f"  Min eigenvalue:        {min_eigenvalue:.6e}")
    print(f"  Negative eigenvalues:  {n_negative} out of {n}")
    print(f"  PSD:                   {is_psd}")
    
    if not is_psd:
        # Show the five most negative eigenvalues so you can
        # gauge how badly PSD is violated
        worst = np.sort(eigenvalues)[:min(5, n)]
        print(f"  Five smallest eigenvalues: {worst}")
    
    print()
    return is_psd, eigenvalues




In [ ]:
check_psd(rings_affinity, "k-NN affinity A")


In [ ]:
check_psd(rings_aff_full, "full affinity A")